# HyperStreamDB: Full Pipeline Presentation

**Version:** 0.1.9 (Alpha)  
**Project:** HyperStreamDB - Serverless Index-Streaming Data Lake

This masterclass takes you from zero to a production-grade RAG pipeline, demonstrating hardware acceleration, hybrid search, and serverless table management.

## 1. Setup & Environment

We initialize our environment and ensure all vector search dependencies are loaded.

In [1]:
import hyperstreamdb as hdb
import pyarrow as pa
import pandas as pd
import numpy as np
import os, shutil, requests, time, warnings
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 150)

print("Environment ready.")

Environment ready.


## 2. Hardware Acceleration

HyperStreamDB can utilize Intel GPUs for massively parallel index construction. We check for available hardware here.

GPU support for Nvidia, Intel, Apple MPS, AMD(ROCm - untested)

In [6]:
import torch
device = "cpu"
if torch.cuda.is_available():
    device = "cuda:0"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")
ctx = hdb.Device(device)


Using device: mps


RuntimeError: Backend 'mps' not available. Build with 'mps' feature.

## 3. Data Preparation (AG News)

We load the AG News dataset and generate semantic embeddings using a 384-dimensional SBERT model.

In [4]:
print("Loading dataset...")
dataset = load_dataset("ag_news", split="test[:500]")
df = pd.DataFrame(dataset)
label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
df["category"] = df["label"].map(label_map)

print("Generating embeddings...")
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["text"].tolist())
df["embedding"] = [list(e) for e in embeddings]
print(f"Prepared {len(df)} records with {len(df['embedding'][0])}D vectors.")

df["id"] = range(len(df))


Loading dataset...
Generating embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prepared 500 records with 384D vectors.


## 4. Table Creation (Serverless Iceberg)

HyperStreamDB tables are serverless and use standard Iceberg/Parquet formats with sidecar indices.

In [5]:
TABLE_URI = "file:///tmp/hdb_masterclass"
if os.path.exists("/tmp/hdb_masterclass"): shutil.rmtree("/tmp/hdb_masterclass")

schema = pa.schema([
    ('id', pa.int32()), 
    ('category', pa.string()),
    ('text', pa.string()),
    ('embedding', pa.list_(pa.float32(), 384))
])

table = hdb.Table.create(TABLE_URI, schema, device=ctx)
print(f"Table created at {TABLE_URI}")

Committed Manifest v1 (Schema Update)
Table created at file:///tmp/hdb_masterclass


DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)
DEBUG: ManifestManager::load_latest: Found version 1 on disk at Path { raw: "_manifest/v1.json" }
DEBUG: ManifestManager::load_latest: Successfully loaded v1 (entries=0)


## 5. Ingestion & Indexing

We write the data and building HNSW vector indices and Inverted scalar indices simultaneously.

In [ ]:
table.add_index_columns(["embedding", "category", "text"])
table.write(df[["id", "category", "text", "embedding"]])
table.commit()
print("Data ingested and indexed successfully.")

## 6. Hybrid Search Performance

We perform a semantic search combined with a scalar filter to find 'Space' news in the 'Sci/Tech' category.

In [ ]:
query_vec = list(model.encode("Latest milestones in space exploration"))

results = (table.filter("category = 'Sci/Tech'")
                 .vector_search(query_vec, k=3)
                 .to_pandas())

print("Hybrid Search Results (Sci/Tech + Vector):")
display(results[["category", "text", "distance"]])

## 7. SQL & pgvector Interface

HyperStreamDB supports standard SQL with pgvector-compatible operators for vector distance.

In [ ]:
session = hdb.Session()
session.register("news", table)

query_vec_str = str(query_vec).replace(' ', '')
sql_df = session.sql(f"""
    SELECT category, text, 
           embedding <-> '{query_vec_str}'::vector AS dist
    FROM news
    WHERE category = 'Sci/Tech'
    ORDER BY dist ASC LIMIT 3
""")
display(sql_df.to_pandas())

## 8. Vector Aggregations

Calculate semantic centroids for entire categories directly in SQL.

In [ ]:
agg_df = session.sql("""
    SELECT category, 
           vector_avg(embedding) AS centroid,
           COUNT(*) as count
    FROM news
    GROUP BY category
    order by count asc
""").to_pandas()
display(agg_df[["category", "count", "centroid"]])

## 9. Table Maintenance

Keep the data lake healthy by compacting small files and vacuuming history.

In [ ]:
print(f"Files before: {len(table.manifest().entries)}")
table.compact()
table.commit()
print(f"Files after: {len(table.manifest().entries)}")

table.vacuum(retention_versions=1)
print("Vacuum complete.")

## 10. Advanced Partitioning

Scale to massive datasets using identity partitioning on the category field.

In [ ]:
PART_URI = "file:///tmp/part_demo"
if os.path.exists("/tmp/part_demo"): shutil.rmtree("/tmp/part_demo")

partition_spec = {
    "fields": [{"name": "category", "transform": "identity", "source_id": 1, "field_id": 1000}]
}

part_table = hdb.Table.create_partitioned(PART_URI, schema, partition_spec)
part_table.write(df[:100])
part_table.commit()

print("Partitioned physical layout:")
for entry in part_table.manifest().entries:
    print(f" - {entry.file_path.split('/')[-2:]}")

## 11. End-to-End RAG Pipeline

Using SQuAD validation data to build a knowledge-backed Q&A system.

In [ ]:
print("Loading SQuAD contexts...")
squad = load_dataset("squad", split="validation[0:100]")
squad_df = pd.DataFrame(squad).drop_duplicates(subset=["context"])

squad_emb = model.encode(squad_df["context"].tolist(), show_progress_bar=True)
squad_df["embedding"] = [list(e) for e in squad_emb]

rag_table = hdb.Table("rag_db_presentation")
rag_table.add_index_columns(["embedding", "context"])
rag_table.write(squad_df[["context", "title", "embedding"]])
rag_table.commit()
print("RAG knowledge base initialized.")

## 12. Inference with Retrieval

Connect retrieval to the LLM (requires GROQ_API_KEY).

In [ ]:
def ask_hdb(question):
    # Convert question to an embedding (using your local model)
    q_emb = model.encode(question)
    
    # Perform a vector search on your table
    # This is where your table data gets pulled in!
    ctx_df = rag_table.to_pandas(vector_filter={
        "column": "embedding", 
        "query": q_emb, 
        "k": 3  # Get top 3 matching contexts
    })
    
    # Combine retrieved contexts into one string
    contexts = ctx_df["context"].tolist()
    prompt = f"Answer the question based ONLY on this context:\n\n" + "\n".join(contexts) + f"\n\nQuestion: {question}"
    
    # Use a supported Groq model
    model_name = "llama-3.3-70b-versatile"
    
    api_key = os.environ.get("GROQ_TOKEN")
    r = requests.post("https://api.groq.com/openai/v1/chat/completions", 
                      headers={"Authorization": f"Bearer {api_key}"},
                      json={"model": model_name, "messages": [{"role": "user", "content": prompt}]})
    
    res = r.json()
    if "choices" not in res:
        # This will show you the exact error from Groq (e.g., "model_not_found")
        print(f"Groq API Error: {res}")
        return "Error from API"
        
    return res["choices"][0]["message"]["content"]

In [ ]:
question = "Where were the first modern Olympic games?"
print(f"Q: {question}\nA: {ask_hdb(question)}")

In [ ]:
question = "What city was superbowl 50 played at?"
print(f"Q: {question}\nA: {ask_hdb(question)}")